# Bachelier's Theory of Speculation Revisited

## Interactive Reproduction Notebook

This notebook reproduces the main numerical experiments from the Alpha Stochastic Research repository:

**Bachelier (1900): Theory of Speculation — Reproducible Python Package for the Origins of Quantitative Finance**

The notebook uses the package interface:

```python
from asr.models import bachelier
```

It covers:

1. Bachelier arithmetic Brownian motion.
2. Martingale and variance-scaling checks.
3. Bachelier European call pricing.
4. Monte Carlo validation.
5. At-the-money square-root-of-time scaling.
6. Local comparison with Black-Scholes.


## 1. Project Setup

Run this notebook either from the repository root or from the `notebooks/` directory.

If the package has not been installed with `pip install -e .`, this setup cell adds `src/` to the Python path so that the notebook still works.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

current_dir = Path.cwd()

if (current_dir / "src").exists():
    ROOT_DIR = current_dir
elif (current_dir.parent / "src").exists():
    ROOT_DIR = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not find the repository root. "
        "Run this notebook from the repository root or from the notebooks/ directory."
    )

SRC_DIR = ROOT_DIR / "src"
FIGURES_DIR = ROOT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repository root: {ROOT_DIR}")
print(f"Source directory: {SRC_DIR}")
print(f"Figures directory: {FIGURES_DIR}")

## 2. Import the Public Package Interface

The notebook imports the same package interface that users see after installation.


In [ ]:
from asr.models import bachelier

print("ASR Bachelier package imported successfully.")
print("Version:", bachelier.__version__)

## 3. Bachelier Arithmetic Brownian Motion

The Bachelier model assumes that the price process follows:

\[
P_t = P_0 + \sigma W_t,
\]

where \(W_t\) is a standard Brownian motion.

This implies:

\[
\mathbb{E}[P_t] = P_0,
\]

and

\[
\mathrm{Var}(P_t) = \sigma^2 t.
\]


In [ ]:
INITIAL_PRICE = 100.0
VOLATILITY = 2.0
MATURITY = 1.0
N_STEPS = 250
N_PATHS = 5_000
SEED = 42

time_grid, paths = bachelier.simulate_paths(
    initial_price=INITIAL_PRICE,
    volatility=VOLATILITY,
    maturity=MATURITY,
    n_steps=N_STEPS,
    n_paths=N_PATHS,
    seed=SEED,
)

analysis = bachelier.analyze_paths(
    time_grid=time_grid,
    paths=paths,
    initial_price=INITIAL_PRICE,
    volatility=VOLATILITY,
)

summary = pd.DataFrame(
    {
        "Quantity": [
            "Initial price",
            "Arithmetic volatility",
            "Maturity",
            "Number of paths",
            "Number of time steps",
            "Terminal empirical mean",
            "Terminal empirical variance",
            "Theoretical terminal variance",
            "Maximum mean deviation from P0",
        ],
        "Value": [
            INITIAL_PRICE,
            VOLATILITY,
            MATURITY,
            N_PATHS,
            N_STEPS,
            analysis.terminal_mean,
            analysis.terminal_empirical_variance,
            analysis.terminal_theoretical_variance,
            analysis.max_mean_deviation,
        ],
    }
)

summary

## 4. Visual Check: Martingale and Variance Scaling

The left panel shows simulated Bachelier price paths.
The right panel compares empirical variance with the theoretical variance \(\sigma^2 t\).


In [ ]:
standard_deviation = np.sqrt(analysis.theoretical_variance)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for path in paths[:25]:
    axes[0].plot(time_grid, path, linewidth=0.8, alpha=0.35)

axes[0].plot(time_grid, analysis.mean_path, linewidth=2.5, label="Empirical mean")
axes[0].plot(
    time_grid,
    INITIAL_PRICE + standard_deviation,
    linestyle="--",
    linewidth=1.5,
    label="P0 +/- one std. dev.",
)
axes[0].plot(time_grid, INITIAL_PRICE - standard_deviation, linestyle="--", linewidth=1.5)

axes[0].set_title("Bachelier price paths")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Price")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_grid, analysis.empirical_variance, linewidth=2.0, label="Empirical variance")
axes[1].plot(
    time_grid,
    analysis.theoretical_variance,
    linestyle="--",
    linewidth=2.0,
    label="Theoretical variance",
)

axes[1].set_title("Variance scaling")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Variance")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("Bachelier arithmetic Brownian motion", fontsize=14)
fig.tight_layout()
plt.show()

## 5. Save the Brownian Motion Figure

This uses the package function and saves the figure in `figures/`.


In [ ]:
brownian_figure_path = bachelier.save_brownian_motion_figure(
    time_grid=time_grid,
    paths=paths,
    analysis=analysis,
    output_path=FIGURES_DIR / "fig1_random_walk_martingale.png",
)

brownian_figure_path

## 6. Bachelier European Call Pricing

For a European call option with strike \(K\) and maturity \(T\), the payoff is:

\[
(P_T - K)^+.
\]

Under the Bachelier model:

\[
P_T = P_0 + \sigma \sqrt{T} Z,
\quad Z \sim \mathcal{N}(0,1).
\]

The Bachelier call price is:

\[
C = (P_0 - K)\Phi(d) + \sigma\sqrt{T}\phi(d),
\]

where:

\[
d = \frac{P_0-K}{\sigma\sqrt{T}}.
\]


In [ ]:
STRIKE = 100.0
OPTION_N_PATHS = 500_000
OPTION_SEED = 7

closed_form_price = bachelier.call_price(
    initial_price=INITIAL_PRICE,
    strike=STRIKE,
    volatility=VOLATILITY,
    maturity=MATURITY,
)

monte_carlo_price, monte_carlo_standard_error = bachelier.call_monte_carlo_price(
    initial_price=INITIAL_PRICE,
    strike=STRIKE,
    volatility=VOLATILITY,
    maturity=MATURITY,
    n_paths=OPTION_N_PATHS,
    seed=OPTION_SEED,
)

pricing_summary = pd.DataFrame(
    {
        "Quantity": [
            "Initial price",
            "Strike",
            "Arithmetic volatility",
            "Maturity",
            "Monte Carlo paths",
            "Closed-form price",
            "Monte Carlo price",
            "Monte Carlo standard error",
            "Absolute difference",
        ],
        "Value": [
            INITIAL_PRICE,
            STRIKE,
            VOLATILITY,
            MATURITY,
            OPTION_N_PATHS,
            closed_form_price,
            monte_carlo_price,
            monte_carlo_standard_error,
            abs(closed_form_price - monte_carlo_price),
        ],
    }
)

pricing_summary

## 7. High-Level Option Pricing Experiment

The package also provides a higher-level reproducibility function.


In [ ]:
bachelier.run_option_pricing_experiment(
    initial_price=INITIAL_PRICE,
    strike=STRIKE,
    volatility=VOLATILITY,
    maturity=MATURITY,
    n_paths=OPTION_N_PATHS,
    seed=OPTION_SEED,
)

## 8. At-the-Money Square-Root-of-Time Scaling

For an at-the-money call where \(K=P_0\), the Bachelier price becomes:

\[
C_{ATM} = \sigma \sqrt{T}\phi(0).
\]

This shows that the at-the-money option value scales with \(\sqrt{T}\).


In [ ]:
maturity_grid = np.linspace(0.01, 2.0, 100)
atm_prices = np.array(
    [
        bachelier.atm_call_price(
            volatility=VOLATILITY,
            maturity=time_to_maturity,
        )
        for time_to_maturity in maturity_grid
    ]
)

theoretical_atm_prices = VOLATILITY * np.sqrt(maturity_grid) / np.sqrt(2.0 * np.pi)

atm_table = pd.DataFrame(
    {
        "Maturity": maturity_grid,
        "Bachelier ATM price": atm_prices,
        "Theoretical ATM price": theoretical_atm_prices,
    }
)

atm_table.head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(maturity_grid, atm_prices, linewidth=2.2, label="Bachelier ATM price")
plt.plot(
    maturity_grid,
    theoretical_atm_prices,
    linestyle="--",
    linewidth=2.0,
    label="sigma sqrt(T) phi(0)",
)
plt.title("ATM option value scaling")
plt.xlabel("Maturity")
plt.ylabel("Call price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9. Local Comparison with Black-Scholes

The Black-Scholes model assumes proportional price changes, while the Bachelier model assumes absolute price changes.

For a local comparison near \(S_0\), we match arithmetic volatility by:

\[
\sigma_{Bach} \approx \sigma_{BS} S_0.
\]


In [ ]:
strike_grid, bachelier_prices, black_scholes_prices = bachelier.compare_with_black_scholes(
    spot=100.0,
    black_scholes_volatility=0.02,
    maturity=1.0,
)

comparison_table = pd.DataFrame(
    {
        "Strike": strike_grid,
        "Bachelier price": bachelier_prices,
        "Black-Scholes price": black_scholes_prices,
        "Difference": bachelier_prices - black_scholes_prices,
    }
)

comparison_table.head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(strike_grid, bachelier_prices, linewidth=2.2, label="Bachelier")
plt.plot(
    strike_grid,
    black_scholes_prices,
    linestyle="--",
    linewidth=2.0,
    label="Black-Scholes",
)
plt.title("Bachelier vs Black-Scholes")
plt.xlabel("Strike")
plt.ylabel("Call price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. Save the Option Pricing Figure

This saves the second figure used in the paper and README.


In [ ]:
option_figure_path = bachelier.save_option_pricing_figure(
    output_path=FIGURES_DIR / "fig2_option_pricing.png",
    initial_price=INITIAL_PRICE,
    bachelier_volatility=VOLATILITY,
    black_scholes_volatility=0.02,
    maturity=MATURITY,
)

option_figure_path

## 11. Reproducibility Checklist

This notebook confirms the main numerical content of the repository:

- The simulated Bachelier process has empirical mean close to \(P_0\).
- Empirical variance follows \(\sigma^2 t\).
- The Bachelier closed-form option price is validated by Monte Carlo simulation.
- At-the-money option prices scale with \(\sqrt{T}\).
- Bachelier and Black-Scholes are locally close under a low-relative-volatility calibration.

For formal validation, run the test suite from the repository root:

```bash
pytest -q
```

For script-based reproduction:

```bash
python src/brownian_motion.py
python src/option_pricing.py
```
